# 04 AIGC Research QA Engine Runner

This notebook executes the Day 6 QA Engine and evaluation suite over the research corpus stored in Google Drive.

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

PROJECT_ROOT = Path("/content/AIGC_Fake_Detection")
DATA_DIR = Path("/content/drive/MyDrive/AIGC/Data")

if not PROJECT_ROOT.exists():
    !git clone -b day6.5-qa-polish https://github.com/IanJ332/AIGC_Fake_Detection.git {PROJECT_ROOT}
else:
    %cd {PROJECT_ROOT}
    !git fetch origin
    !git checkout day6.5-qa-polish
    !git pull origin day6.5-qa-polish

%cd {PROJECT_ROOT}
os.environ['PYTHONPATH'] = str(PROJECT_ROOT)

Mounted at /content/drive
Cloning into '/content/AIGC_Fake_Detection'...
remote: Enumerating objects: 203, done.
remote: Counting objects: 100% (203/203), done.
remote: Compressing objects: 100% (151/151), done.
remote: Total 203 (delta 86), reused 159 (delta 47), pack-reused 0 (from 0)
Receiving objects: 100% (203/203), 960.00 KiB | 3.68 MiB/s, done.
Resolving deltas: 100% (86/86), done.
/content/AIGC_Fake_Detection


## 2. Dependencies

In [2]:
!pip install pandas tqdm duckdb

## 3. Sample Queries
Verify the CLI and operators are working.

In [3]:
!python -m src.query.cli --data-dir {DATA_DIR} --question "What are the top 10 datasets mentioned across the corpus?"

----------------------------------------
QUESTION: What are the top 10 datasets mentioned across the corpus?
TIER: aggregation
----------------------------------------
Answer:
Top 10 dataset entities across the corpus:
1. ImageNet (mentioned in 306 locations)
2. GenImage (mentioned in 247 locations)
3. ArtiFact (mentioned in 202 locations)
4. LAION (mentioned in 164 locations)
5. COCO (mentioned in 122 locations)
6. WildFake (mentioned in 102 locations)
7. Celeb-DF (mentioned in 64 locations)
8. FakeBench (mentioned in 51 locations)
9. DiffusionDB (mentioned in 44 locations)
10. CIFAKE (mentioned in 43 locations)


Evidence:
- P003 (GenImage: A Million-Scale Benchmark for Detecting AI-Generated Image, 2023.0)
  Anchor: P003:p1-p2:introduction
  Snippet: 1 Introduction arXiv:2306.08571v2  [cs.CV]  24 Jun 2023 The advancement of generative models has yielded remarkable progress in synthesizing photorealistic images, drastically reducing the requisite expertise and effort to generate fake

In [4]:
!python -m src.query.cli --data-dir {DATA_DIR} --question "What does paper P001 propose?"

----------------------------------------
QUESTION: What does paper P001 propose?
TIER: single_doc
----------------------------------------
Answer:
Findings for P001 (Detecting GAN generated Fake Images using Co-occurrence Matrices, 2019.0):
- Datasets: nan
- Models: CNN
- Metrics: accuracy
- Generators: GAN
- Method Keywords: co-occurrence
- Results Extracted: 0

Evidence:
- P001 (Detecting GAN generated Fake Images using Co-occurrence Matrices, 2019.0)
  Anchor: abstract
  Snippet: Back to articles Articles Volume: 31 | Article ID: art00008 [image] Detecting GAN generated Fake Images using Co- occurrence Matrices •  Image forensics GAN image detection Fake news Multimedia security Deep Learning Fake images Lakshmanan Nataraj Tajuddin Manhar Mohammed B. S. Manjunath Shivkumar Chandrasekaran Arjuna Flenner Jawadul H. Bappy Amit K. Roy-Chowdhury   DOI :  10.2352/ISSN.2470-1173.2019.5.MWSF-532  Published Online :  January 2019 [image] [image] [image] Abstract The advent of Gen...
- P001 (

## 4. Full Evaluation
Run all 40 questions and generate the summary report.

In [5]:
!python eval/run_eval.py --data-dir {DATA_DIR} --questions eval/questions_40.jsonl

Running evaluation on eval/questions_40.jsonl...
Evaluation complete. Summary saved to eval/results/day6_eval_summary.md


## 5. Sync Results to Drive

In [6]:
!mkdir -p {DATA_DIR}/reports/day6_5_eval
!cp -r {PROJECT_ROOT}/eval/results/* {DATA_DIR}/reports/day6_5_eval/
print(f"Results synced to {DATA_DIR}/reports/day6_5_eval/")

Results synced to /content/drive/MyDrive/AIGC/Data/reports/day6_5_eval/


## 6. Audit Summary

In [7]:
summary_path = DATA_DIR / "reports" / "day6_eval" / "day6_eval_summary.md"
if summary_path.exists():
    with open(summary_path, 'r') as f:
        print(f.read())
else:
    print("Summary report not found. Check evaluation logs.")

# Day 6 QA Engine Evaluation Summary

- **Total Questions**: 40
- **Tier Routing Accuracy**: 37/40 (92.5%)
- **Unknown Tier Count**: 0
- **Operator Execution Success**: 40/40
- **Answers with Evidence**: 21
- **Answers with Limitations**: 40

## Per-Tier Distribution
| Tier | Count |
| :--- | :--- |
| single_doc | 5 |
| aggregation | 6 |
| contradiction | 5 |
| temporal | 5 |
| citation_graph | 4 |
| multihop | 5 |
| negation | 5 |
| quantitative | 5 |

